# Notebook 02: GIS foundation

**Project:** Perishable Inventory Optimizer (GIS + supply chain)
**Milestone:** 2, GIS foundation

## What we are trying to accomplish
Give the stores a real place on the map. The sales data only says which city and state each store is in. We add coordinates, Ecuador's boundaries and assumed warehouse sites, then assign every store to a warehouse and draw the service areas. The stocking recommendation is made per warehouse, so this step decides what "region" means in the rest of the project.

## What we got accomplished
Steps 1 to 3 are done: the 54 stores sit in 22 cities, each city has coordinates saved in `data/reference/city_coordinates.csv`, and the map check found and corrected one wrong match (Libertad). Step 4 (province boundaries and a province check) is written and ready to run.

## What we have left to do
Finish Step 4, then choose assumed warehouse sites, assign each store to a warehouse, and draw the service-area map.

**How each step works:** we run the code, look at the result, and fill in the explanation cells together: what we are looking for, how to read it, what it means, and what is next and why.

## Step 1: Which cities do the stores sit in?

### What we are looking for
The list of distinct cities, the state each belongs to, and how many stores are in each. This is the full set of places we have to put on the map, and it shows how concentrated the business is. A city with many stores matters more to a warehouse decision than a city with one.

In [21]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")
stores = pd.read_csv(next(RAW.rglob("stores.csv")))

cities = (
    stores.groupby(["state", "city"])
    .agg(stores=("store_nbr", "count"))
    .reset_index()
    .sort_values(["stores", "city"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Stores: {len(stores)}  |  Cities: {len(cities)}  |  States: {stores['state'].nunique()}")
display(cities)

Stores: 54  |  Cities: 22  |  States: 16


,state,city,stores
0,Pichincha,Quito,18
1,Guayas,Guayaquil,8
2,Azuay,Cuenca,3
3,Santo Domingo de los Tsachilas,Santo Domingo,3
4,Tungurahua,Ambato,2
5,Cotopaxi,Latacunga,2
6,El Oro,Machala,2
7,Manabi,Manta,2
8,Los Rios,Babahoyo,1
9,Pichincha,Cayambe,1


### How to read it
Each row is one city, with its state and the number of stores there. The list is sorted so the biggest concentrations come first. Each city name belongs to exactly one state, so there are no duplicate names to untangle. The line above the table gives the totals: 54 stores, 22 cities, 16 states.

### What it means
The business is concentrated. Quito (18 stores) and Guayaquil (8) together hold 26 of the 54 stores, 48%, and they are Ecuador's two biggest cities. Any warehouse plan will be dominated by them; a Quito warehouse alone would serve a third of the chain. The other 28 stores are spread across 20 smaller cities, which is where distance and geography start to matter.

Several names are ambiguous on a map. "Salinas", "Libertad", "El Carmen" and "Santo Domingo" each exist in more than one place. That is why we keep the state alongside the city when we look up coordinates.

### What's next and why
Step 2 gives each of the 22 cities a latitude and longitude. Nothing else on the map can happen without them.

## Step 2: Where is each city?

### What we are looking for
A latitude and longitude for each of the 22 cities. We ask the free OpenStreetMap geocoder (through the `geopy` package) for "city, state, Ecuador" and keep the answer, the address it matched, and the query we used. Then we check that every point falls inside Ecuador's rough outline. A wrong match, such as a "Salinas" in another country, would put a store in the wrong place and quietly spoil every distance we calculate later.

The results are saved to `data/reference/city_coordinates.csv`, so the lookup runs once and the project can be repeated without asking the geocoder again. Delete that file to look the cities up afresh. The map data is © OpenStreetMap contributors (ODbL licence).

In [22]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

COORDS_FILE = Path("../data/reference/city_coordinates.csv")

# OpenStreetMap spells a few places differently from the sales data.
SEARCH_NAME = {"Libertad": "La Libertad"}

# A rough rectangle around mainland Ecuador, used only as a sanity check.
LAT_MIN, LAT_MAX = -5.1, 1.5
LON_MIN, LON_MAX = -81.2, -75.1

if COORDS_FILE.exists():
    coords = pd.read_csv(COORDS_FILE)
    print(f"Loaded saved coordinates from {COORDS_FILE} (delete the file to look them up again).")
else:
    geolocator = Nominatim(user_agent="gis-supply-chain-perishable-learning-project")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)  # polite: about 1 request per second

    rows = []
    for r in cities.itertuples():
        name = SEARCH_NAME.get(r.city, r.city)
        hit, used = None, None
        for query in (f"{name}, {r.state}, Ecuador", f"{name}, Ecuador"):
            hit = geocode(query, country_codes="ec")
            if hit:
                used = query
                break
        rows.append({
            "state": r.state,
            "city": r.city,
            "stores": r.stores,
            "latitude": hit.latitude if hit else None,
            "longitude": hit.longitude if hit else None,
            "query_used": used,
            "matched_address": hit.address if hit else None,
        })
    coords = pd.DataFrame(rows)
    COORDS_FILE.parent.mkdir(parents=True, exist_ok=True)
    coords.to_csv(COORDS_FILE, index=False)
    print(f"Looked up {len(coords)} cities and saved them to {COORDS_FILE}.")

coords["inside_ecuador_box"] = (
    coords["latitude"].between(LAT_MIN, LAT_MAX) & coords["longitude"].between(LON_MIN, LON_MAX)
)

print(f"Cities found: {coords['latitude'].notna().sum()} of {len(coords)}")
print(f"Inside Ecuador's outline: {coords['inside_ecuador_box'].sum()} of {len(coords)}")
display(coords[["city", "state", "stores", "latitude", "longitude", "inside_ecuador_box", "matched_address"]])

Loaded saved coordinates from ..\data\reference\city_coordinates.csv (delete the file to look them up again).
Cities found: 22 of 22
Inside Ecuador's outline: 22 of 22


,city,state,stores,latitude,longitude,inside_ecuador_box,matched_address
0,Quito,Pichincha,18,-0.220164,-78.512327,True,"Quito, Pichincha, 170130, Ecuador"
1,Guayaquil,Guayas,8,-2.190057,-79.886867,True,"Guayaquil, Guayas, Ecuador"
2,Cuenca,Azuay,3,-2.897407,-79.004173,True,"Cuenca, Azuay, 010101, Ecuador"
3,Santo Domingo,Santo Domingo de los Tsachilas,3,-0.247503,-79.171317,True,"Santo Domingo, Santo Domingo de los Tsachilas,..."
4,Ambato,Tungurahua,2,-1.242241,-78.628759,True,"Ambato, Tungurahua, Ecuador"
5,Latacunga,Cotopaxi,2,-0.934031,-78.614576,True,"Latacunga, Cotopaxi, Ecuador"
6,Machala,El Oro,2,-3.314894,-79.951280,True,"Machala, El Oro, Ecuador"
7,Manta,Manabi,2,-1.032006,-80.816931,True,"Manta, Manabí, Ecuador"
8,Babahoyo,Los Rios,1,-1.876260,-79.506892,True,"Babahoyo, Los Ríos, Ecuador"
9,Cayambe,Pichincha,1,-0.003601,-78.047623,True,"Cayambe, Pichincha, Ecuador"


### How to read it
Each row is a city with its latitude (north or south of the equator, negative meaning south) and longitude (negative meaning west). `inside_ecuador_box` says whether the point falls in a rough rectangle around Ecuador. `matched_address` is the place OpenStreetMap actually found, which makes it the most useful column for catching mistakes.

### What it means
All 22 cities were found and all 22 passed the rectangle check. Most results look right: Quito is at -0.22, -78.51, Guayaquil at -2.19, -79.89 and Cuenca at -2.90, -79.00, which match those cities.

The rectangle check misses one error. **Libertad matched "La Libertad, Abel Gilbert, Parroquia Febres Cordero, Guayaquil"**, a neighbourhood inside Guayaquil. The real La Libertad is a coastal city about 100 km west, next to Salinas. Both places are inside Ecuador, so the point passes the check. Kept as it is, that store would sit about 100 km from its true location and distort every distance we calculate later.

The sales file lists Libertad under Guayas, but La Libertad has belonged to Santa Elena province since 2007. That is a quirk in the source data, and we record it rather than change the original file.

A few other points may sit away from the town centre, because the geocoder sometimes returns the centre of the wider area: Manta (the longitude may put it just offshore), El Carmen, Guaranda, Ibarra and Cayambe. These are judged from general knowledge of Ecuador's geography and are checked on a map in Step 3.

### What's next and why
Step 3 plots the 22 points on a real map so mistakes show up, and corrects Libertad by searching for "La Libertad, Santa Elena, Ecuador".

## Step 3: Are the cities in the right places?

### What we are looking for
Each city plotted on a real map, so we can see at a glance whether any point is in the wrong place. The automatic rectangle check cannot catch a point that is in Ecuador but in the wrong spot, as with Libertad. We also correct Libertad and save the corrected coordinates, keeping a record of what was changed and why.

The map uses OpenStreetMap tiles, so it needs an internet connection. Each map is also saved as a web page in `docs/figures/` that can be opened in a browser.

In [23]:
import plotly.express as px

FIGURES = Path("../docs/figures")
FIGURES.mkdir(parents=True, exist_ok=True)


def draw_city_map(df, title):
    """Draw the store cities on an OpenStreetMap background.

    Takes a table with `city`, `state`, `stores`, `latitude` and `longitude`
    columns and returns a Plotly figure. Each circle is one city, sized by its
    number of stores; hover over a circle to see its details.
    """
    return px.scatter_map(
        df,
        lat="latitude",
        lon="longitude",
        size="stores",
        size_max=28,
        text="city",
        hover_name="city",
        hover_data={"state": True, "stores": True, "latitude": ":.3f", "longitude": ":.3f"},
        center={"lat": -1.5, "lon": -78.5},
        zoom=5.2,
        map_style="open-street-map",
        title=title,
        height=700,
    )


# 3a. The cities as the geocoder first placed them
fig = draw_city_map(coords, "Store cities as geocoded (before correction)")
fig.write_html(FIGURES / "city_map_before_correction.html", include_plotlyjs="cdn")
fig.show()

In [24]:
# 3b. Correct Libertad.
# OpenStreetMap matched "Libertad, Guayas" to a neighbourhood of Guayaquil. The real
# La Libertad is in Santa Elena province, next to Salinas, so we search for that instead.
FIXES = {"Libertad": "La Libertad, Santa Elena, Ecuador"}

geolocator = Nominatim(user_agent="gis-supply-chain-perishable-learning-project")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

salinas = coords.loc[coords["city"] == "Salinas"].iloc[0]

for city, query in FIXES.items():
    hit = geocode(query, country_codes="ec")
    assert hit is not None, f"No match found for {query!r}"

    # The corrected point should be close to Salinas (within about 0.3 degrees, roughly 30 km).
    assert abs(hit.longitude - salinas["longitude"]) < 0.3 and abs(hit.latitude - salinas["latitude"]) < 0.3, (
        f"{query!r} matched {hit.address!r}, which is not near Salinas"
    )

    row = coords["city"] == city
    coords.loc[row, ["latitude", "longitude"]] = [hit.latitude, hit.longitude]
    coords.loc[row, ["query_used", "matched_address"]] = [query, hit.address]
    print(f"{city}: now {hit.latitude:.4f}, {hit.longitude:.4f}")
    print(f"  matched: {hit.address}")

coords.drop(columns="inside_ecuador_box").to_csv(COORDS_FILE, index=False)
print(f"Saved the corrected coordinates to {COORDS_FILE}")

# Redraw the map with the correction
fig = draw_city_map(coords, "Store cities after correcting Libertad")
fig.write_html(FIGURES / "city_map_after_correction.html", include_plotlyjs="cdn")
fig.show()

Libertad: now -2.2207, -80.9087
  matched: La Libertad, Santa Elena, Ecuador
Saved the corrected coordinates to ..\data\reference\city_coordinates.csv


### How to read it
Each circle is a city, sized by its number of stores. Before the correction, Libertad sat in the Guayaquil area. After it, the circle is on the Santa Elena peninsula, next to Salinas, where La Libertad really is.

### What it means
The geocoder matched "Libertad, Guayas" to a neighbourhood of Guayaquil. The sales file lists Libertad under Guayas, but La Libertad has been in Santa Elena province since 2007. The wrong point would have put a store in the wrong warehouse area later, so we corrected it by hand and the code checks the new point is near Salinas.

### What's next and why
The map check only catches errors we can see by eye. Step 4 adds real province boundaries so the computer can check all 22 cities, not just the ones that look odd.

## Step 4. Do the cities fall in the right provinces?

### What we are looking for
Each city should sit inside the province the sales file gives for it. We load Ecuador's 24 provinces from Natural Earth (public domain, saved in `data/external/`) and find the province each point falls in. Any city that lands in a different province has a wrong point.

In [ ]:
import unicodedata
import geopandas as gpd

PROVINCE_FILE = Path("../data/external/ne_10m_admin_1_states_provinces/ne_10m_admin_1_states_provinces.shp")
VERSION_FILE = PROVINCE_FILE.with_suffix(".VERSION.txt")
METRIC_CRS = "EPSG:32717"  # UTM zone 17S: distances and areas come out in metres


def plain(text):
    """Lower-case a place name and strip its accents, so "Manabí" matches "Manabi"."""
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode().lower().strip()


world = gpd.read_file(PROVINCE_FILE)
print(f"Natural Earth Admin 1, version {VERSION_FILE.read_text().strip()}: {len(world):,} regions worldwide")

provinces = world.loc[world["admin"] == "Ecuador", ["name", "iso_3166_2", "check_me", "geometry"]].copy()
provinces["area_km2"] = (provinces.to_crs(METRIC_CRS).area / 1e6).round(0)
print(f"Ecuador: {len(provinces)} provinces")

# Two provinces have a problem we can see in the file itself: Napo should be about
# 12,500 km2 and Tungurahua about 3,300 km2, but their sizes here are the other way round.
# Both rows also carry the dataset's own "check me" flag.
display(provinces.loc[provinces["name"].isin(["Napo", "Tungurahua"]), ["name", "area_km2", "check_me"]])

size = provinces.set_index("name")["area_km2"]
if size["Napo"] < size["Tungurahua"]:
    provinces["name"] = provinces["name"].replace({"Napo": "Tungurahua", "Tungurahua": "Napo"})
    print("Swapped the names of Napo and Tungurahua back (their sizes showed they were reversed).")
size = provinces.set_index("name")["area_km2"]
assert size["Napo"] > 10_000 and size["Tungurahua"] < 5_000, "Napo and Tungurahua still look wrong"

# Do the province names in the boundary file match the states in the sales file?
state_names = set(stores["state"].map(plain))
province_names = set(provinces["name"].map(plain))
print(f"\nStates in the sales file: {len(state_names)}")
print("Missing from the boundary file:", sorted(state_names - province_names) or "none")
print("Provinces with no store:", sorted(provinces.loc[~provinces["name"].map(plain).isin(state_names), "name"]))

### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [ ]:
# The sales file puts Libertad in Guayas, but La Libertad has been in Santa Elena province since 2007.
# The point is right (Step 3), so we expect Santa Elena here.
EXPECTED_PROVINCE = {"Libertad": "Santa Elena"}


def check_provinces(coords):
    """Find the province each city's point falls in and compare it with the sales file.

    Takes a table with `city`, `state`, `latitude` and `longitude` columns. A point just
    outside a province, such as a port city on a simplified coastline, is matched to the
    nearest province within 5 km. Returns the table with `province`, `km_outside` (0 means
    inside) and `agrees` (True when the province matches the expected one).
    """
    coords = coords.drop(columns=[c for c in ("province", "km_outside") if c in coords.columns])
    points = gpd.GeoDataFrame(
        coords,
        geometry=gpd.points_from_xy(coords["longitude"], coords["latitude"]),
        crs="EPSG:4326",
    ).to_crs(METRIC_CRS)
    polygons = provinces.to_crs(METRIC_CRS)[["name", "geometry"]].rename(columns={"name": "province"})
    found = gpd.sjoin_nearest(points, polygons, how="left", max_distance=5_000, distance_col="metres_outside")
    found = found[~found.index.duplicated()]
    out = pd.DataFrame(found.drop(columns=["geometry", "index_right", "metres_outside"]))
    out["km_outside"] = (found["metres_outside"] / 1000).round(1)
    expected = out["city"].map(EXPECTED_PROVINCE).fillna(out["state"]).map(plain)
    out["agrees"] = expected == out["province"].map(lambda p: plain(p) if isinstance(p, str) else "")
    return out


result = check_provinces(coords)
print(f"Cities in their expected province: {result['agrees'].sum()} of {len(result)}")
display(result[["city", "state", "province", "km_outside", "agrees"]])

### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [ ]:
# Cities that fall in the wrong province are looked up again. The first lookup can return the
# centre of the wider area (the canton) instead of the town, so we ask for city-type and
# settlement-type places. Each new point is accepted only if it falls in the expected province.
MANUAL_FALLBACK = {"Ibarra": (0.3517, -78.1223)}  # approximate town centre; used only if the lookups fail

geolocator = Nominatim(user_agent="gis-supply-chain-perishable-learning-project")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

to_fix = result.loc[~result["agrees"], "city"].tolist()
print("Cities to look up again:", to_fix or "none")

for city in to_fix:
    row = coords["city"] == city
    state = coords.loc[row, "state"].iloc[0]
    fixed = False

    for kind in ("city", "settlement"):
        hit = geocode(f"{city}, {state}, Ecuador", country_codes="ec", featuretype=kind)
        if hit is None:
            continue
        trial = coords.loc[row].copy()
        trial[["latitude", "longitude"]] = [hit.latitude, hit.longitude]
        if check_provinces(trial)["agrees"].iloc[0]:
            coords.loc[row, ["latitude", "longitude"]] = [hit.latitude, hit.longitude]
            coords.loc[row, ["query_used", "matched_address"]] = [f"{city}, {state}, Ecuador ({kind})", hit.address]
            fixed = True
            break

    if not fixed and city in MANUAL_FALLBACK:
        lat, lon = MANUAL_FALLBACK[city]
        trial = coords.loc[row].copy()
        trial[["latitude", "longitude"]] = [lat, lon]
        if check_provinces(trial)["agrees"].iloc[0]:
            coords.loc[row, ["latitude", "longitude"]] = [lat, lon]
            coords.loc[row, ["query_used", "matched_address"]] = ["manual: approximate town centre", None]
            fixed = True

    assert fixed, f"Could not place {city} in its province"
    print(f"{city}: now {coords.loc[row, 'latitude'].iloc[0]:.4f}, {coords.loc[row, 'longitude'].iloc[0]:.4f}"
          f" via {coords.loc[row, 'query_used'].iloc[0]}")

result = check_provinces(coords)
assert result["agrees"].all(), "Some cities are still in the wrong province"
print(f"\nCities in their expected province: {result['agrees'].sum()} of {len(result)}")

result[["state", "city", "stores", "latitude", "longitude", "province", "km_outside", "query_used", "matched_address"]].to_csv(
    COORDS_FILE, index=False
)
print(f"Saved the checked coordinates, with their provinces, to {COORDS_FILE}")
display(result[["city", "state", "province", "km_outside", "agrees"]])

### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)

In [ ]:
import plotly.graph_objects as go


def draw_province_map(df, title):
    """Draw Ecuador's provinces with the store cities on top.

    Takes the table of cities (`city`, `stores`, `latitude`, `longitude`) and returns a
    Plotly figure: provinces as pale shapes with dark outlines, cities as circles sized
    by their number of stores.
    """
    shapes = provinces[["name", "geometry"]].set_index("name")
    fig = go.Figure()
    fig.add_trace(go.Choroplethmap(
        geojson=shapes.__geo_interface__,
        locations=shapes.index,
        z=[1] * len(shapes),
        colorscale=[[0, "#8fb996"], [1, "#8fb996"]],
        showscale=False,
        marker_opacity=0.25,
        marker_line_width=1.5,
        marker_line_color="#355e3b",
        hovertext=shapes.index,
        hoverinfo="text",
    ))
    fig.add_trace(go.Scattermap(
        lat=df["latitude"],
        lon=df["longitude"],
        mode="markers+text",
        text=df["city"],
        textposition="top right",
        textfont={"color": "#222222", "size": 12},
        marker={"size": 8 + 3 * df["stores"] ** 0.5, "color": "#2a78d6", "opacity": 0.8},
        customdata=df[["stores"]],
        hovertemplate="%{text}<br>%{customdata[0]} store(s)<extra></extra>",
    ))
    fig.update_layout(
        title=title,
        height=700,
        margin={"l": 0, "r": 0, "t": 50, "b": 0},
        showlegend=False,
        map={"style": "open-street-map", "center": {"lat": -1.5, "lon": -78.5}, "zoom": 5.2},
    )
    return fig


fig = draw_province_map(result, "Store cities and Ecuador's provinces")
fig.write_html(FIGURES / "city_map_with_provinces.html", include_plotlyjs="cdn")
fig.show()

### How to read it
(Fill in together after running.)

### What it means
(Fill in together after running.)

### What's next and why
(Fill in together after running.)